In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

In [2]:
df_train = pd.read_csv("data/train_final.csv")

In [3]:
df_valid = pd.read_csv("data/valid_final.csv")

In [4]:
X_train = df_train.drop(columns=['cust_id', 'revenue_2018_2019'])

In [5]:
y_train = df_train['revenue_2018_2019']

In [8]:
X_val = df_valid.drop(columns=['cust_id', 'revenue_2018_2019'])
y_val = df_valid['revenue_2018_2019']

In [9]:
categorical_cols = ['prod_brand_mode']

In [10]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

In [11]:
y_train_binary = (y_train > 0).astype(int)

In [12]:
mask_spenders = y_train > 0
X_train_spenders = X_train[mask_spenders]
y_train_spenders = y_train[mask_spenders]

In [13]:
print("Training Classifier (Probability of Return)...")
clf_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    class_weight='balanced' # Helps the model since 64% of data is class 0
)

clf_model.fit(X_train, y_train_binary)

Training Classifier (Probability of Return)...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 34076, number of negative: 59196
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1914
[LightGBM] [Info] Number of data points in the train set: 93272, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.05
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [14]:
val_probabilities = clf_model.predict_proba(X_val)[:, 1]

In [22]:
will_return_mask = val_probabilities > 0.50

In [15]:
print("Training Regressor (Expected Spend)...")
reg_model = lgb.LGBMRegressor(
    random_state=42,
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    objective='mae' # Keeping the magic bullet!
)

Training Regressor (Expected Spend)...


In [16]:
reg_model.fit(X_train_spenders, y_train_spenders)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000253 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1812
[LightGBM] [Info] Number of data points in the train set: 34076, number of used features: 11
[LightGBM] [Info] Start training from score 129.945007
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.05
,n_estimators,150
,subsample_for_bin,200000
,objective,'mae'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [17]:
val_predicted_spend = reg_model.predict(X_val)

In [23]:
final_predictions = np.zeros_like(val_predicted_spend)

In [18]:
val_predicted_spend = np.clip(val_predicted_spend, a_min=0, a_max=None)

In [24]:
final_predictions[will_return_mask] = val_predicted_spend[will_return_mask]

In [25]:
mae = mean_absolute_error(y_val, final_predictions)
spearman_corr, p_value = spearmanr(y_val, final_predictions)

In [26]:
print("-" * 30)
print(f"Two-Stage Validation MAE: %{mae:.2f}")
print(f"Two-Stage Spearman Correlation: {spearman_corr:.4f}")
print("-" * 30)

------------------------------
Two-Stage Validation MAE: %71.52
Two-Stage Spearman Correlation: 0.3966
------------------------------
